In [ ]:
# Librerias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

In [ ]:
# Parametros

umbral_pico = 0.4
maq_obj = 'PLE1'

tol_pct = 0.05 # tolerancia para determinar si la señal esta dentro de la media 
t_estable = pd.Timedelta(seconds=10) # Tiempo minimo que tiene que pasar la señal cerca de la media para considerar que el evento termino

FASES = ['R', 'S', 'T']

In [ ]:
# Leer rutas

from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "data"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("Existe data:", DATA_PATH.exists())


PROJECT_ROOT: c:\Users\HP Spectre X360\Desktop\MARIANA\COLLOQUIA_2025\Analisis_Powermeter
DATA_PATH: c:\Users\HP Spectre X360\Desktop\MARIANA\COLLOQUIA_2025\Analisis_Powermeter\data
Existe data: True


In [ ]:
# Abrir los graficos en el navegador

pio.renderers.default = 'browser'

In [ ]:
# Funcion para cargar datos

from pathlib import Path
import pandas as pd

def cargar_maquina(base_path, maquina):
    base_path = Path(base_path).resolve()
    paths = list((base_path / maquina).rglob("*.csv"))

    if not paths:
        raise ValueError(f"No se encontraron CSV para {maquina} en {base_path}")

    dfs = []
    for path in paths:
        df = pd.read_csv(path)
        df["maquina"] = maquina
        dfs.append(df)

    return (
        pd.concat(dfs, ignore_index=True)
          .sort_values("temporal_placa")
          .reset_index(drop=True)
    )


In [ ]:
# Carga de datos

df_ple1 = cargar_maquina(DATA_PATH, "PLE1")
df_ple7 = cargar_maquina(DATA_PATH, "PLE7")

In [ ]:
# Funcion preparar_df

def preparar_df(df):
    df = df.copy()

    # Timestamp
    df['temporal_placa'] = pd.to_datetime(df['temporal_placa'])
    df['hora'] = df['temporal_placa'].dt.hour
    df['minuto'] = df['temporal_placa'].dt.minute

    # Turnos
    def asignar_turno(hora, minuto):
        t = hora * 60 + minuto

        # Pausas
        if 12*60 <= t < 12*60 + 30:
            return 'ALMUERZO'
        if 22*60 <= t < 22*60 + 30:
            return 'CENA'

        # Turnos
        if 5*60 <= t < 17*60:
            return 'TURNO MAÑANA'
        if 17*60 <= t < 22*60:
            return 'TURNO TARDE'
        if (t >= 22*60 + 30) or (t < 1*60):
            return 'TURNO TARDE'

        return 'FUERA_TURNO'

    df['turno'] = df.apply(
        lambda x: asignar_turno(x['hora'], x['minuto']),
        axis=1
    )
        
    # Potencias totales por timestamp
    df['p_activa_total'] = (
        df['potencia_a_r'] +
        df['potencia_a_s'] +
        df['potencia_a_t']
    )

    df['q_reactiva_total'] = (
        df['potencia_r_r'] +
        df['potencia_r_s'] +
        df['potencia_r_t']
    )

    return df


In [ ]:
# Ajuste de datos de las plegadoras

df_ple1 = preparar_df(df_ple1)
df_ple7 = preparar_df(df_ple7)

df_all = pd.concat([df_ple1, df_ple7], ignore_index=True)

# Filtra los datos correspondientes al 9-1-2026
import datetime as dt

dia = dt.date(2026, 1, 12)
df_all = df_all[df_all['temporal_placa'].dt.date == dia]

In [ ]:
# Calculo la media de las corrientes excluyendo los picos

def media_sin_picos(s, q = 0.9):
    return s[s <= s.quantile(q)].mean()

medias_por_fase = (
    df_all[df_all['maquina'] == maq_obj]
    .assign(
        R=df_all['corriente_r'],
        S=df_all['corriente_s'],
        T=df_all['corriente_t'],
    )
    .melt(
        id_vars=['maquina'],
        value_vars=['R', 'S', 'T'],
        var_name='fase',
        value_name='corriente'
    )
    .groupby(['maquina', 'fase'], as_index=False)
    .agg(
        baseline=('corriente', media_sin_picos)
    )
)


In [ ]:
# Funcion detectar_picos

def detectar_picos(signal, timestamp, umbral_pico):
    dt = timestamp.diff().dt.total_seconds()
    valid = dt > 0

    dI_dt = signal.diff() / dt

    return (
        valid &
        dI_dt.notna() &
        (dI_dt > umbral_pico)
    )

In [ ]:
# Detección de picos en las 3 fases

import pandas as pd

maquinas = df_all['maquina'].unique()

fase_cols = {
    'R': 'corriente_r',
    'S': 'corriente_s',
    'T': 'corriente_t'
}

df_peaks_list = []

for maq in maquinas:

    g = (
        df_all[df_all['maquina'] == maq]
        .sort_values('temporal_placa')
        .copy()
    )

    if g.empty:
        continue

    for fase, col_corriente in fase_cols.items():

        señal = g[col_corriente]
        timestamp = g['temporal_placa']

        idx_picos = detectar_picos(señal, timestamp, umbral_pico)

        if len(idx_picos) == 0:
            continue

        df_peaks = g.loc[idx_picos, ['temporal_placa']].copy()
        df_peaks['maquina'] = maq
        df_peaks['fase'] = fase    

        df_peaks_list.append(df_peaks)

# DataFrame final de picos
df_peaks_all = pd.concat(df_peaks_list, ignore_index=True)

df_peaks_all['temporal_placa'] = (
    pd.to_datetime(df_peaks_all['temporal_placa'], errors='coerce')
    .dt.tz_localize(None)
)


In [ ]:
# Visualizacion de picos detectados y su marca temporal

df_peaks_all = df_peaks_all.sort_values(
    by=['maquina', 'fase']
).reset_index(drop=True)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

df_peaks_all

,temporal_placa,maquina,fase
0,2026-01-12 05:00:36,PLE1,R
1,2026-01-12 05:00:38,PLE1,R
2,2026-01-12 05:20:18,PLE1,R
3,2026-01-12 05:20:31,PLE1,R
4,2026-01-12 05:20:36,PLE1,R
5,2026-01-12 05:20:51,PLE1,R
6,2026-01-12 05:23:30,PLE1,R
7,2026-01-12 05:23:40,PLE1,R
8,2026-01-12 05:24:03,PLE1,R
9,2026-01-12 05:26:55,PLE1,R


In [ ]:
# Funcion detectar_fin_evetos

def detectar_fin_por_retorno(df_signal, t_ini, baseline, tol, t_estable):
    """
    df_signal: DataFrame con columnas ['temporal_placa', 'corriente']
    t_ini: timestamp desde donde buscar el fin
    baseline: valor medio de corriente
    tol: tolerancia absoluta
    """
    df_post = df_signal[df_signal['temporal_placa'] >= t_ini].copy()

    df_post['en_regimen'] = (
        (df_post['corriente'] >= baseline - tol) &
        (df_post['corriente'] <= baseline + tol)
    )

    # detectar permanencia continua
    df_post['dt'] = df_post['temporal_placa'].diff()
    df_post['grupo'] = (
        (~df_post['en_regimen']) |
        (df_post['dt'] > df_post['dt'].median() * 2)
    ).cumsum()

    for _, g in df_post.groupby('grupo'):
        if g['en_regimen'].all():
            dur = g['temporal_placa'].iloc[-1] - g['temporal_placa'].iloc[0]
            if dur >= t_estable:
                return g['temporal_placa'].iloc[0]

    return None


In [ ]:
# Analisis de eventos

import pandas as pd

# =================================================
# 1) PREPROCESAR SEÑALES
# =================================================

signal_dict = {}

df_all_ple1 = df_all[df_all['maquina'] == maq_obj].copy()

df_all_ple1['temporal_placa'] = (
    pd.to_datetime(df_all_ple1['temporal_placa'])
    .dt.tz_localize(None)
)

for fase in FASES:
    col = f'corriente_{fase.lower()}'
    signal_dict[fase] = (
        df_all_ple1[['temporal_placa', col]]
        .rename(columns={col: 'corriente'})
        .sort_values('temporal_placa')
        .reset_index(drop=True)
    )

# =================================================
# 2) BASELINE POR FASE (ROBUSTO)
# =================================================

baseline_fase = (
    medias_por_fase
    .set_index('fase')['baseline']
    .to_dict()
)

# =================================================
# 3) PICOS DE LA MAQUINA
# =================================================

df_peaks = df_peaks_all[
    df_peaks_all['maquina'] == maq_obj
].copy()

df_peaks['temporal_placa'] = (
    pd.to_datetime(df_peaks['temporal_placa'])
    .dt.tz_localize(None)
)

# =================================================
# 4) DETECCION DE EVENTOS (SIN SOLAPES)
# =================================================

rows = []

for fase in FASES:

    fase = str(fase).upper().strip()

    if fase not in baseline_fase:
        continue

    baseline = baseline_fase[fase]
    tol = baseline * tol_pct

    df_signal = signal_dict[fase]

    df_peaks_fase = (
        df_peaks[df_peaks['fase'].str.upper() == fase]
        .sort_values('temporal_placa')
        .reset_index(drop=True)
    )

    if df_peaks_fase.empty:
        continue

    # lookup rápido de picos
    picos_set = set(df_peaks_fase['temporal_placa'])

    # ---------------------------
    # ESTADOS
    # ---------------------------
    regimen_confirmado = False
    t_regimen_inicio_global = None

    evento_activo = False
    evento_id = 0
    fecha_inicio = None

    t_regimen_inicio = None

    # =================================================
    # LOOP SOBRE LA SEÑAL CONTINUA
    # =================================================
    for i in range(len(df_signal)):

        t = df_signal.loc[i, 'temporal_placa']
        corriente = df_signal.loc[i, 'corriente']

        hay_pico = t in picos_set
        en_regimen = (baseline - tol <= corriente <= baseline + tol)

        # ------------------------------------------------
        # 4.1 WARM-UP: confirmar régimen inicial
        # ------------------------------------------------
        if not regimen_confirmado:
            if en_regimen:
                if t_regimen_inicio_global is None:
                    t_regimen_inicio_global = t
                elif t - t_regimen_inicio_global >= t_estable:
                    regimen_confirmado = True
            else:
                t_regimen_inicio_global = None
            continue

        # ------------------------------------------------
        # 4.2 INICIO DE EVENTO
        # ------------------------------------------------
        if not evento_activo:
            if hay_pico:
                evento_activo = True
                evento_id += 1
                fecha_inicio = t
                t_regimen_inicio = None
            continue

        # ------------------------------------------------
        # 4.3 EVENTO ACTIVO
        # ------------------------------------------------
        if not en_regimen:
            # fuera de régimen → sigue evento
            t_regimen_inicio = None
            continue

        if hay_pico:
            # pico mientras vuelve → no puede cerrar
            t_regimen_inicio = None
            continue

        # en régimen, sin pico
        if t_regimen_inicio is None:
            t_regimen_inicio = t
        elif t - t_regimen_inicio >= t_estable:
            # FIN REAL DEL EVENTO
            rows.append({
                'maquina': maq_obj,
                'fase': fase,
                'ID_Evento': evento_id,
                'Fecha Inicio': fecha_inicio,
                'Fecha Fin': t_regimen_inicio,
                'Duracion Evento': t_regimen_inicio - fecha_inicio
            })
            evento_activo = False
            fecha_inicio = None
            t_regimen_inicio = None

    # ------------------------------------------------
    # EVENTO ABIERTO SIN CIERRE (final del dataset)
    # ------------------------------------------------
    if evento_activo and fecha_inicio is not None:
        t_fin = df_signal.iloc[-1]['temporal_placa']
        rows.append({
            'maquina': maq_obj,
            'fase': fase,
            'ID_Evento': evento_id,
            'Fecha Inicio': fecha_inicio,
            'Fecha Fin': t_fin,
            'Duracion Evento': t_fin - fecha_inicio
        })

# =================================================
# 5) DATAFRAME FINAL
# =================================================

df_evento = (
    pd.DataFrame(rows)
    .sort_values(by=['fase', 'ID_Evento'])
    .reset_index(drop=True)
)

df_evento


,maquina,fase,ID_Evento,Fecha Inicio,Fecha Fin,Duracion Evento
0,PLE1,R,1,2026-01-12 05:20:18,2026-01-12 05:21:09,0 days 00:00:51
1,PLE1,R,2,2026-01-12 05:23:30,2026-01-12 05:24:11,0 days 00:00:41
2,PLE1,R,3,2026-01-12 05:26:55,2026-01-12 05:27:23,0 days 00:00:28
3,PLE1,R,4,2026-01-12 05:30:55,2026-01-12 05:32:18,0 days 00:01:23
4,PLE1,R,5,2026-01-12 05:35:52,2026-01-12 05:36:30,0 days 00:00:38
5,PLE1,R,6,2026-01-12 05:40:25,2026-01-12 05:40:58,0 days 00:00:33
6,PLE1,R,7,2026-01-12 05:45:16,2026-01-12 05:45:48,0 days 00:00:32
7,PLE1,R,8,2026-01-12 05:47:52,2026-01-12 05:48:22,0 days 00:00:30
8,PLE1,R,9,2026-01-12 05:50:34,2026-01-12 05:50:36,0 days 00:00:02
9,PLE1,R,10,2026-01-12 05:50:51,2026-01-12 05:51:17,0 days 00:00:26


In [ ]:
# Eventos a .cvs
from pathlib import Path

OUT_TABLAS = PROJECT_ROOT / "output" / "tablas"
OUT_TABLAS.mkdir(parents=True, exist_ok=True)

df_evento.to_csv(OUT_TABLAS / "df_eventos.csv", index=False)

In [ ]:
# Graficas de eventos por fase 

import plotly.graph_objects as go
from plotly.subplots import make_subplots

def xref_subplot(row):
    return 'x' if row == 1 else f'x{row}'

def yref_subplot(row):
    return 'y domain' if row == 1 else f'y{row} domain'


map_fase_col = {
    'R': 'corriente_r',
    'S': 'corriente_s',
    'T': 'corriente_t'
}

colores_fase = {
    'R': 'red',
    'S': 'green',
    'T': 'blue'
}

maquinas = sorted(df_all['maquina'].unique())

# Figura con 2 filas (una por máquina)
fig = make_subplots(
    rows=len(maquinas),
    cols=1,
    shared_xaxes=False,
    subplot_titles=[f'Máquina {m}' for m in maquinas]
)

# Guardamos índices de traces por (maquina, fase)
trace_idx = {}

# -----------------
# Corrientes
# -----------------
for row, maq in enumerate(maquinas, start=1):

    df_m = df_all[df_all['maquina'] == maq]

    for fase, col in map_fase_col.items():

        visible = (fase == 'R')  # fase inicial

        fig.add_trace(
            go.Scatter(
            x=df_m['temporal_placa'],
            y=df_m[col],
            mode='lines',
            name=f'Fase {fase}',          # nombre genérico
            line=dict(color=colores_fase[fase]),
            visible=visible,
            showlegend=(row == 3)         
        ),
    row=row,
    col=1
)


        trace_idx[(maq, fase)] = len(fig.data) - 1

# Shapes (eventos) por fase
shapes_por_fase = {}

for fase in map_fase_col.keys():

    shapes = []

    for row, maq in enumerate(maquinas, start=1):

        df_e = df_evento[
            (df_evento['maquina'] == maq) &
            (df_evento['fase'] == fase)
        ]

        for _, ev in df_e.iterrows():

            shapes.append(
                dict(
                    type='rect',
                     xref=xref_subplot(row),    
                    yref=yref_subplot(row),
                    x0=ev['Fecha Inicio'],
                    x1=ev['Fecha Fin'],
                    y0=0,
                    y1=1,
                    fillcolor=colores_fase[fase],
                    opacity=0.15,
                    line_width=0
                )
            )

    shapes_por_fase[fase] = shapes

# Dropdown único
botones = []

for fase in map_fase_col.keys():

    visibles = [False] * len(fig.data)

    for maq in maquinas:
        visibles[trace_idx[(maq, fase)]] = True

    botones.append(
        dict(
            label=f'Fase {fase}',
            method='update',
            args=[
                {'visible': visibles},
                {'shapes': shapes_por_fase[fase]}
            ]
        )
    )

# Layout final
fig.update_layout(
    updatemenus=[
        dict(
            buttons=botones,
            direction='down',
            x=1.08,
            y=1,
            showactive=True
        )
    ],
    title='Eventos de corriente por maquina',
    shapes=shapes_por_fase['R']  # fase inicial
)
fig.update_yaxes(title_text='Corriente [A]', row=1, col=1)
fig.update_yaxes(title_text='Corriente [A]', row=2, col=1)

fig.show()


In [ ]:
# Combinacion de fases superpuestas por maquina
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def xref_subplot(row):
    return 'x' if row == 1 else f'x{row}'

def yref_subplot(row):
    return 'y domain' if row == 1 else f'y{row} domain'

# Configuración
map_fase_col = {
    'R': 'corriente_r',
    'S': 'corriente_s',
    'T': 'corriente_t'
}

colores_fase = {
    'R': 'red',
    'S': 'green',
    'T': 'blue'
}

maquinas = sorted(df_all['maquina'].unique())

# Figura con subplots (una fila por máquina)
fig = make_subplots(
    rows=len(maquinas),
    cols=1,
    shared_xaxes=False,
    subplot_titles=[f'Máquina {m}' for m in maquinas]
)


# Corrientes (fases superpuestas)
for row, maq in enumerate(maquinas, start=1):

    df_m = df_all[df_all['maquina'] == maq]

    for fase, col in map_fase_col.items():

        fig.add_trace(
            go.Scatter(
                x=df_m['temporal_placa'],
                y=df_m[col],
                mode='lines',
                name=f'Fase {fase}',
                line=dict(color=colores_fase[fase]),
                showlegend=(row == 1)  # una sola leyenda
            ),
            row=row,
            col=1
        )


# Shapes: eventos por fase y máquina
shapes = []

for fase in map_fase_col.keys():

    for row, maq in enumerate(maquinas, start=1):

        df_e = df_evento[
            (df_evento['maquina'] == maq) &
            (df_evento['fase'] == fase)
        ]

        for _, ev in df_e.iterrows():

            shapes.append(
                dict(
                    type='rect',
                    xref=xref_subplot(row),
                    yref=yref_subplot(row),
                    x0=ev['Fecha Inicio'],
                    x1=ev['Fecha Fin'],
                    y0=0,
                    y1=1,
                    fillcolor=colores_fase[fase],
                    opacity=0.15,
                    line_width=0
                )
            )

# Layout final
fig.update_layout(
    title='Eventos de corriente por máquina (fases superpuestas)',
    shapes=shapes
)

# Títulos de eje Y en todos los subplots
for r in range(1, len(maquinas) + 1):
    fig.update_yaxes(title_text='Corriente [A]', row=r, col=1)

fig.show()
